# Regression - clim_sens and sqrt

In this notebook we create many regression models - baselines and GPSR.

We generate results files.

This notebook requires the output from the data cleaning and data exploration notebooks.

The *main* regression notebook gives main results. This notebook tries some alternate versions: climate sensitivity is truncated and we introduce sqrt to the language.


In [28]:
from pysr import PySRRegressor
import Magpie
#print(dir(Magpie))
from Magpie import MagpieRegressor
import time
import numpy as np
print(np.__version__)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import sympy as sp
import os
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.linear_model import ElasticNet


from sklearn.ensemble import RandomForestRegressor

#from pysr import PySRRegressor  # import after env var is set
import matplotlib




matplotlib.rcParams['pdf.fonttype'] = 42 # helps produce ACM-compliant figures
matplotlib.rcParams['ps.fonttype'] = 42
Xy = pd.read_csv(r"C:\Users\Sophie\OneDrive - National University of Ireland, Galway\Y1S2\Thesis\GECCO Code\Outputs\data_Xy_clim_sens_filter.csv", index_col=0)
Xy.head()


2.1.3


,Scenario,SOW,Temp_Limit,Delay,GDP,Pop,Other_ESD_Drivers,SDR,Elast_ESD_Driver,Elast_ESD_Price,...,GSupply_Geothermal,GSupply_Nuclear,GSupply_Solar,GSupply_Tidal_Waves,GSupply_Wind,GConsumption,GConsumption_Fossil,GConsumption_Nuclear,GConsumption_Renewable,GCost
0,BASE_SSP2,1,8.0,0,1.02329,1.009280,1.063100,3.70267,1.013020,1.216860,...,1.907061,4.167604,5.115586,2.424417,5.161512,812.092289,539.893683,43.075844,229.122762,24.788425
4,BASE_SSP2,5,8.0,0,1.00481,0.997576,1.127530,3.46723,1.087940,1.118800,...,1.915827,4.226128,5.810294,2.516766,6.502266,874.573547,592.973597,43.678409,237.921540,26.175837
5,BASE_SSP2,6,8.0,0,1.08269,1.031770,0.993713,3.77124,0.873656,0.989665,...,1.899880,4.272520,3.658103,2.384190,4.454559,712.984178,462.390198,44.156060,206.437920,22.522647
6,BASE_SSP2,7,8.0,0,1.11940,0.989002,0.971626,4.54539,1.055970,0.752712,...,1.932460,4.401001,5.408875,2.474168,6.264400,897.256776,632.347658,45.478900,219.430218,27.429091
7,BASE_SSP2,8,8.0,0,1.02483,1.022250,0.893182,3.54405,1.023780,0.985672,...,1.906197,4.145143,4.865129,2.404867,6.567324,810.309151,541.500007,42.844589,225.964555,24.229503


In [29]:
features = [
 'Pop',
 'Other_ESD_Drivers',
 'SDR',
 'Elast_ESD_Driver',
 'Elast_ESD_Price',
 'CO2_Storage_Poten',
 'Wind_Poten',
 'Solar_Poten',
 'Biomass_Poten',
 'Oil_Gas_Poten',
 'Solar_PV_Inv_Cost',
 'Wind_Inv_Cost',
 'Bioenergy_CCS_Inv_Cost',
 'Other_Tech_Cost',
 'Forcing',
 'Land_Sinks',
 'Clim_Sens',
 'Year']


## Experimental Design

We try several target variables (ie dependent variable), several subsets of independent variables, several regression models. 

## Cross-validation

In preliminary experiments we see large differences in performance according to train-test split - therefore, we use a 5-fold cross-validation.

## Reporting

For each of those 5 folds, we report the train $R^2$ and test $R^2$.

In [30]:
# We have two sets of X-variable subsets: 'small' and 'all'
# The reason for this is some of the variables are not super impactful on the value of the prediction
# For each subset, we have a variant which includes the scenario-encoding variables (_inc_enc)
# and a variant which excludes them.
# temperature limit and delay are the two features that make up each scenario
# The idea is to compare a small versus a large subset for each scenario.
# And then as an alternative, put all the scenarios together, with the scenario-encoding
# variables included, and compare small versus large.
xvar_subsets = {
    'small': ['SDR', 'Clim_Sens', 'Pop', 'GDP'],
    #'small_inc_enc': ['Temp_Limit', 'Delay', 'SDR', 'Clim_Sens', 'Pop', 'GDP'],
    'all': ['GDP', 'Pop',
       'Other_ESD_Drivers', 'SDR', 'Elast_ESD_Driver', 'Elast_ESD_Price',
       'CO2_Storage_Poten', 'Wind_Poten', 'Solar_Poten', 'Biomass_Poten',
       'Oil_Gas_Poten', 'Solar_PV_Inv_Cost', 'Wind_Inv_Cost',
       'Bioenergy_CCS_Inv_Cost', 'Other_Tech_Cost', 'Forcing', 'Land_Sinks',
       'Clim_Sens'],
    #all_inc_enc': ['Temp_Limit', 'Delay', 'GDP', 'Pop',
     #  'Other_ESD_Drivers', 'SDR', 'Elast_ESD_Driver', 'Elast_ESD_Price',
      # 'CO2_Storage_Poten', 'Wind_Poten', 'Solar_Poten', 'Biomass_Poten',
      # 'Oil_Gas_Poten', 'Solar_PV_Inv_Cost', 'Wind_Inv_Cost',
     #  'Bioenergy_CCS_Inv_Cost', 'Other_Tech_Cost', 'Forcing', 'Land_Sinks',
     #  'Clim_Sens']
}

scenarios = ['BASE_SSP2']#, '1p5c_OS_SSP2', '2C_SSP2', '2C_SSP2_DA30']#removed 'all'

targets = ['GCost'] # cannot do marginal CO2 as it's all-NaNs at 2050 ----- 'GSupply', 'CO2eq', 'GConsumption', 


print(type(xvar_subsets['small']))
print(xvar_subsets.keys())



<class 'list'>
dict_keys(['small', 'all'])


In [ ]:
%debug
import warnings
import os
warnings.filterwarnings("ignore")
from datetime import datetime

def run_everything(Xy):
    import Magpie
    from Magpie import MagpieRegressor
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.neural_network import MLPRegressor
    from pysr import PySRRegressor
    import numpy as np
    from sklearn.metrics import mean_squared_error
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import itertools
                                                                                              
    output_file = f"output_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"                  
    

    ######################################################################################
    # Helper function to generate output log and add date and  timestamps to statements 
    ######################################################################################
    with open(output_file, 'w', encoding='utf-8') as log_file:
        def print_and_log(message, include_timestamp=False):
            """Helper function to print to console and write to log file"""                     
            if include_timestamp:
                timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                formatted_message = f"[{timestamp}]\n {message}"
            else:
                formatted_message = str(message)
            print(formatted_message)
            log_file.write(formatted_message + '\n')
            log_file.flush()
        

        ###############################################################################
        # PARAMETER SWEEP CONFIGURATION
        # Define parameter combinations for Magpie and PySR
        ###############################################################################
       
        max_evals = 1000000
        init_mults = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]          
        init_evals = [int(max_evals * f) for f in init_mults]  # [100, 500, 1000, 5000]

        magpie_params = [(max_evals, x) for x in init_evals]


        
        print_and_log(f"\n{'='*80}")
        print_and_log(f"PARAMETER SWEEP CONFIGURATION")
        print_and_log(f"{'='*80}")



        ###############################################################################
        # Main loop through targets x scenarios x Xvar subsets
        # Each combination trains models with different parameter configurations
        ###############################################################################
        results = []
        for target in targets:
            for scenario in scenarios:
                if scenario == 'ALL':
                    Xy_sub = Xy
                else: 
                    Xy_sub = Xy[Xy['Scenario'] == scenario]
     
                for xvar_subset_code in xvar_subsets:
                    # for the 'ALL' scenario, run only if we have _inc_enc
                    if scenario == 'ALL' and (not "_inc_enc" in xvar_subset_code): 
                        continue
                    # for the non-ALL scenarios, run only if we DO NOT have _inc_enc
                    if scenario != 'ALL' and ("_inc_enc" in xvar_subset_code): 
                        continue

                    xvar_subset = xvar_subsets[xvar_subset_code]
                    X = Xy_sub[xvar_subset]
                    y = Xy_sub[target]
                    print_and_log(f'PARAMETERS: Scenario {scenario}, xvar code {xvar_subset_code} xvars {xvar_subset}, target {target}\n')
                    print_and_log("")

                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

                    ###############################################################################
                    # MAGPIE PARAMETER SWEEP
                    ###############################################################################
                    for max_evals, init_evals in magpie_params:
                        initevals =  init_evals
                        maxevals = max_evals
                        print(f"Init Evals = {initevals}, Max Evals = {maxevals}")

                        model = MagpieRegressor(
                            maxevals=maxevals,
                            #initevals=initevals
                        )
                        model_name = f'Magpie_max{maxevals}'#_init{initevals}'
                        
                        print_and_log(f"TRAINING: {model_name}...\n", include_timestamp=True)
                        
                        try:
                            model.fit(X_train, y_train)
                            
                            # Predictions
                            y_pred_train = model.predict(X_train)
                            y_pred_test = model.predict(X_test)

                            print_and_log(f"DEBUG AFTER PREDICT - {model_name}:", include_timestamp=True)
                            print_and_log(f"  type(y_pred_train): {type(y_pred_train)}")
                            
                            if not hasattr(y_pred_train, 'shape'):
                                print_and_log(f"  ERROR: y_pred_train is not an array! Value: {y_pred_train}")
                                continue
                        
                            ################################################################################################
                            # Calculate metrics
                            ################################################################################################
                            from sklearn.metrics import r2_score, mean_absolute_error
                                
                            mse_train = mean_squared_error(y_train, y_pred_train)
                            mse_test = mean_squared_error(y_test, y_pred_test)
                            rmse_train = np.sqrt(mse_train)
                            rmse_test = np.sqrt(mse_test)
                            mae_train = mean_absolute_error(y_train, y_pred_train)
                            mae_test = mean_absolute_error(y_test, y_pred_test)
                            r2_train = r2_score(y_train, y_pred_train)
                            r2_test = r2_score(y_test, y_pred_test)
                            
                            print_and_log(f"\n{'='*60}")
                            print_and_log(f"MODEL QUALITY METRICS: {model_name}", include_timestamp=True)
                            print_and_log(f"{'='*60}")
                            print_and_log(f"Training Set Performance:")
                            print_and_log(f"  MSE:  {mse_train:.6f}")
                            print_and_log(f"  RMSE: {rmse_train:.6f}")
                            print_and_log(f"  MAE:  {mae_train:.6f}")
                            print_and_log(f"  R²:   {r2_train:.6f}")
                            print_and_log(f"\nTest Set Performance:")
                            print_and_log(f"  MSE:  {mse_test:.6f}")
                            print_and_log(f"  RMSE: {rmse_test:.6f}")
                            print_and_log(f"  MAE:  {mae_test:.6f}")
                            print_and_log(f"  R²:   {r2_test:.6f}")
                            print_and_log(f"\nOverfitting Check:")
                            print_and_log(f"  Train-Test MSE Ratio: {mse_train/mse_test:.4f}")
                            print_and_log(f"  Train-Test R² Diff:   {r2_train - r2_test:.4f}")

                            ############################################################################################
                            # Model-specific statistics: Top 5 equations for Magpie
                            ############################################################################################
                            print_and_log(f"\n{'='*60}")
                            print_and_log(f"TOP 5 EQUATIONS (Magpie Regressor):")
                            print_and_log(f"{'='*60}")
                            try:
                                top_equations = model.equations_.sort_values(by='loss_validation').head(5)
                                print_and_log(top_equations[["size", "loss", "loss_validation", "equation"]])
                                best_row = model.equations_.loc[model.equations_["loss_validation"].idxmin()] 
                                best_equation = str(best_row['equation'])
                                print_and_log(f"\nBest equation: {best_equation}")
                            except Exception as e:
                                print_and_log(f"Could not retrieve equations: {e}")
                                best_equation = "Equation not available"

                            ############################################################################################
                            # Store results for parameter sweep analysis
                            ############################################################################################
                            results.append({
                                'model_type': 'Magpie',
                                'model_name': model_name,
                                'scenario': scenario,
                                'target': target,
                                'xvar_subset': xvar_subset_code,
                                'param1': maxevals,
                                #'param2': initevals,
                                'mse_train': mse_train,
                                'mse_test': mse_test,
                                'r2_train': r2_train,
                                'r2_test': r2_test,
                                'best_equation': best_equation
                            })
                      
                        except Exception as e:
                            print_and_log(f"EXCEPTION in {model_name}: {str(e)}", include_timestamp=True)
                            import traceback
                            print_and_log(traceback.format_exc())
                            continue

                    # PySR




                    ###############################################################################
                    # PYSR PARAMETER SWEEP
                    ###############################################################################
                    #pysr_niterations = [10, 20, 40, 80, 100]  # default is 40
                    

                    #for niterations in pysr_niterations:
                    model = PySRRegressor(
                        max_evals= max_evals
                    )
                    model_name = f'PySR_niter{max_evals}'
                    
                    print_and_log(f"TRAINING: {model_name}...\n", include_timestamp=True)
                    
                    try:
                        model.fit(X_train, y_train)
                        
                        # Predictions
                        y_pred_train = model.predict(X_train)
                        y_pred_test = model.predict(X_test)

                        print_and_log(f"DEBUG AFTER PREDICT - {model_name}:", include_timestamp=True)
                        print_and_log(f"  type(y_pred_train): {type(y_pred_train)}")
                        
                        if not hasattr(y_pred_train, 'shape'):
                            print_and_log(f"  ERROR: y_pred_train is not an array! Value: {y_pred_train}")
                            continue

                        ########################################################################
                        # Calculate metrics
                        ########################################################################
                        from sklearn.metrics import r2_score, mean_absolute_error
                            
                        mse_train = mean_squared_error(y_train, y_pred_train)
                        mse_test = mean_squared_error(y_test, y_pred_test)
                        rmse_train = np.sqrt(mse_train)
                        rmse_test = np.sqrt(mse_test)
                        mae_train = mean_absolute_error(y_train, y_pred_train)
                        mae_test = mean_absolute_error(y_test, y_pred_test)
                        r2_train = r2_score(y_train, y_pred_train)
                        r2_test = r2_score(y_test, y_pred_test)
                        
                        print_and_log(f"\n{'='*60}")
                        print_and_log(f"MODEL QUALITY METRICS: {model_name}", include_timestamp=True)
                        print_and_log(f"{'='*60}")
                        print_and_log(f"Training Set Performance:")
                        print_and_log(f"  MSE:  {mse_train:.6f}")
                        print_and_log(f"  RMSE: {rmse_train:.6f}")
                        print_and_log(f"  MAE:  {mae_train:.6f}")
                        print_and_log(f"  R²:   {r2_train:.6f}")
                        print_and_log(f"\nTest Set Performance:")
                        print_and_log(f"  MSE:  {mse_test:.6f}")
                        print_and_log(f"  RMSE: {rmse_test:.6f}")
                        print_and_log(f"  MAE:  {mae_test:.6f}")
                        print_and_log(f"  R²:   {r2_test:.6f}")
                        print_and_log(f"\nOverfitting Check:")
                        print_and_log(f"  Train-Test MSE Ratio: {mse_train/mse_test:.4f}")
                        print_and_log(f"  Train-Test R² Diff:   {r2_train - r2_test:.4f}")

                        ########################################################################
                        # Top 5 equations
                        ########################################################################
                        print_and_log(f"\n{'='*60}")
                        print_and_log(f"TOP 5 EQUATIONS (PySR):")
                        print_and_log(f"{'='*60}")
                        try:
                            top_equations = model.equations_.sort_values(by='loss').head(5)
                            print_and_log(top_equations[["size", "loss", "equation"]])
                            best_row = model.equations_.loc[model.equations_["loss"].idxmin()]
                            best_equation = str(best_row['equation'])
                            print_and_log(f"\nBest equation: {best_equation}")
                        except Exception as e:
                            print_and_log(f"Could not retrieve equations: {e}")
                            best_equation = "Equation not available"

                        ########################################################################
                        # Store results
                        ########################################################################
                        results.append({
                            'model_type': 'PySR',          
                            'model_name': model_name,
                            'scenario': scenario,
                            'target': target,
                            'xvar_subset': xvar_subset_code,
                            'param1': 10000,               # max_evals (fixed)
                            'param2': #niterations,         # ✅ meaningful sweep parameter
                            'mse_train': mse_train,
                            'mse_test': mse_test,
                            'r2_train': r2_train,
                            'r2_test': r2_test,
                            'best_equation': best_equation
                        })

                    except Exception as e:
                        print_and_log(f"EXCEPTION in {model_name}: {str(e)}", include_timestamp=True)
                        import traceback
                        print_and_log(traceback.format_exc())
                        continue




                            
        ###############################################################################
        # PARAMETER SWEEP VISUALIZATION
        # Generate plots comparing parameter performance
        ###############################################################################
        print_and_log(f"\n{'='*80}")
        print_and_log(f"GENERATING PARAMETER SWEEP VISUALIZATIONS", include_timestamp=True)
        print_and_log(f"{'='*80}\n")
        
        results_df = pd.DataFrame(results)
        
        # Create output directory for parameter sweep plots
        param_sweep_dir = "parameter_sweep_analysis"
        os.makedirs(param_sweep_dir, exist_ok=True)
        
        # Plot 1: Heatmap for each model type
        for model_type in ['Magpie', 'PySR']:
            model_data = results_df[results_df['model_type'] == model_type]
            
            if len(model_data) == 0:
                continue
            
            # Create pivot tables for heatmap
            pivot_mse = model_data.pivot_table(
                values='mse_test',
                index='param2',
                columns='param1',
                aggfunc='mean'
            )
            
            pivot_r2 = model_data.pivot_table(
                values='r2_test',
                index='param2',
                columns='param1',
                aggfunc='mean'
            )
            
            # Plot MSE and R² heatmaps
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
            
            sns.heatmap(pivot_mse, annot=True, fmt='.4f', cmap='RdYlGn_r', ax=ax1)
            param1_label = 'maxevals' if model_type == 'Magpie' else 'max_evals'
            param2_label = 'initevals' if model_type == 'Magpie' else 'niterations'
            ax1.set_title(f'{model_type} - Test MSE by Parameters')
            ax1.set_xlabel(param1_label)
            ax1.set_ylabel(param2_label)
            
            sns.heatmap(pivot_r2, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax2)
            ax2.set_title(f'{model_type} - Test R² by Parameters')
            ax2.set_xlabel(param1_label)
            ax2.set_ylabel(param2_label)
            
            plt.tight_layout()
            plt.savefig(f"{param_sweep_dir}/{model_type}_parameter_heatmap.pdf")
            plt.savefig(f"{param_sweep_dir}/{model_type}_parameter_heatmap.jpg")
            plt.close()
            
            print_and_log(f"Saved heatmap for {model_type}")
        
        # Plot 2: Line plots showing parameter impact
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        for idx, model_type in enumerate(['Magpie', 'PySR']):
            model_data = results_df[results_df['model_type'] == model_type]
            
            if len(model_data) == 0:
                continue
            
            param1_label = 'maxevals' if model_type == 'Magpie' else 'max_evals'
            param2_label = 'initevals' if model_type == 'Magpie' else 'niterations'
            
            # MSE vs param1
            for param2_val in model_data['param2'].unique():
                subset = model_data[model_data['param2'] == param2_val]
                grouped = subset.groupby('param1')['mse_test'].mean()
                axes[idx, 0].plot(grouped.index, grouped.values, marker='o', label=f'{param2_label}={param2_val}')
            
            axes[idx, 0].set_xlabel(param1_label)
            axes[idx, 0].set_ylabel('Test MSE')
            axes[idx, 0].set_title(f'{model_type} - MSE vs {param1_label}')
            axes[idx, 0].legend()
            axes[idx, 0].grid(True, alpha=0.3)
            
            # R² vs param1
            for param2_val in model_data['param2'].unique():
                subset = model_data[model_data['param2'] == param2_val]
                grouped = subset.groupby('param1')['r2_test'].mean()
                axes[idx, 1].plot(grouped.index, grouped.values, marker='o', label=f'{param2_label}={param2_val}')
            
            axes[idx, 1].set_xlabel(param1_label)
            axes[idx, 1].set_ylabel('Test R²')
            axes[idx, 1].set_title(f'{model_type} - R² vs {param1_label}')
            axes[idx, 1].legend()
            axes[idx, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{param_sweep_dir}/parameter_line_plots.pdf")
        plt.savefig(f"{param_sweep_dir}/parameter_line_plots.jpg")
        plt.close()
        
        print_and_log(f"Saved line plots")
        
        # Plot 3: Best performing parameters summary
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        for idx, model_type in enumerate(['Magpie', 'PySR']):
            model_data = results_df[results_df['model_type'] == model_type].copy()
            
            if len(model_data) == 0:
                continue
            
            # Sort by test MSE
            model_data = model_data.sort_values('mse_test')
            top_10 = model_data.head(10)
            
            # Create labels with parameter values
            top_10 = top_10.sort_values('param2')  # sort by initeval (param2) low to high
            labels = [f"{int(row['param1'])},{int(row['param2'])}" for _, row in top_10.iterrows()]
            
            axes[idx].barh(range(len(top_10)), top_10['mse_test'])
            axes[idx].set_yticks(range(len(top_10)))
            axes[idx].set_yticklabels(labels)
            axes[idx].set_xlabel('Test MSE')
            param1_label = 'maxevals' if model_type == 'Magpie' else 'max_evals'
            param2_label = 'initevals' if model_type == 'Magpie' else 'niterations'
            axes[idx].set_ylabel(f'Parameters ({param1_label}, {param2_label})')
            axes[idx].set_title(f'{model_type} - Top 10 Parameter Combinations (by MSE)')
            axes[idx].invert_yaxis()
            
        plt.tight_layout()
        plt.savefig(f"{param_sweep_dir}/top_parameters.pdf")
        plt.savefig(f"{param_sweep_dir}/top_parameters.jpg")
        plt.close()
        
        print_and_log(f"Saved top parameters bar chart")
        
        # Save results to CSV
        results_df.to_csv(f"{param_sweep_dir}/parameter_sweep_results.csv", index=False)
        print_and_log(f"Saved results CSV to {param_sweep_dir}/parameter_sweep_results.csv")
        
        # Print best parameters
        print_and_log(f"\n{'='*80}")
        print_and_log(f"BEST PARAMETERS BY MODEL TYPE")
        print_and_log(f"{'='*80}")
        
        for model_type in ['Magpie', 'PySR']:
            model_data = results_df[results_df['model_type'] == model_type]
            if len(model_data) > 0:
                best = model_data.loc[model_data['mse_test'].idxmin()]
                param1_label = 'maxevals' if model_type == 'Magpie' else 'max_evals'
                param2_label = 'initevals' if model_type == 'Magpie' else 'niterations'
                print_and_log(f"\n{model_type}:")
                print_and_log(f"  Best {param1_label}: {best['param1']}")
                print_and_log(f"  Best {param2_label}: {best['param2']}")
                print_and_log(f"  Test MSE: {best['mse_test']:.6f}")
                print_and_log(f"  Test R²: {best['r2_test']:.6f}")
                print_and_log(f"  Scenario: {best['scenario']}, Target: {best['target']}, Subset: {best['xvar_subset']}")
        
        print_and_log(f"\n{'='*80}")
        print_and_log(f"All parameter sweep plots saved to: {os.path.abspath(param_sweep_dir)}")
        print_and_log(f"{'='*80}\n")
        
        # Log completion
        print_and_log(f"\n{'='*80}\nLog file saved to: {output_file}\n{'='*80}")


# Run the analysis
run_everything(Xy)

> c:\users\sophie\appdata\local\temp\ipykernel_21976\750043507.py(4)<module>()
      2 #results = run_everything(Xy, cv)
      3 results_cols = ['Target', 'Scenario', 'xvar_subset_code', 'Regression Model', 'Fold', 'Train Score', 'Test Score']
----> 4 results_df = pd.DataFrame(results, columns=results_cols)
      5 results_df['Scenario / Variables'] = results_df['Scenario'] + ' / ' + results_df['xvar_subset_code']
      6 #results_df.to_csv(f'../outputs/results_cv5_EN_RF_2xGP_clim_sens_filter_sqrt.csv')


PARAMETER SWEEP CONFIGURATION
PARAMETERS: Scenario BASE_SSP2, xvar code small xvars ['SDR', 'Clim_Sens', 'Pop', 'GDP'], target GCost


Init Evals = 100, Max Evals = 10000
[2026-03-04 20:32:06]
 TRAINING: Magpie_max10000_init100...

n_vars 4
1000
3000
4000
5000
6000
7000
8000
[2026-03-04 20:32:44]
 DEBUG AFTER PREDICT - Magpie_max10000_init100:
  type(y_pred_train): <class 'numpy.ndarray'>

[2026-03-04 20:32:44]
 MODEL QUALITY METRICS: Magpie_max10000_init100
Training Set Performance:


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.791e+00  0.000e+00  y = 25.194
3           2.233e+00  1.115e-01  y = SDR - -21.23
5           2.186e+00  1.072e-02  y = SDR + (GDP + 20.156)
7           2.028e+00  3.740e-02  y = (SDR + (GDP * 8.121)) - -12.507
9           2.028e+00  1.318e-04  y = (((SDR + 1.9921) + GDP) * GDP) + 17.638
11          2.026e+00  4.055e-04  y = ((GDP * 1.0604) * (SDR + (GDP + 1.4613))) + 17.786
13          2.022e+00  1.078e-03  y = (((SDR + 1.4613) + (GDP * Pop)) * (1.0604 * GDP)) + 17...
                                      .786
17          2.022e+00  3.968e-05  y = ((Pop + (SDR - (GDP * -1.7507))) + (((GDP * 5.2451) + ...
                                      GDP) + 7.8709)) / 0.85019
───────────────────────────────────────────────────────────────────────────────────────────────────
[2026-03-04 20:36:23]
 DEBUG AFTER PREDICT - PySR_niter10:
  type(

[ Info: Started!
[ Info: Final population:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.791e+00  0.000e+00  y = 25.194
3           2.233e+00  1.115e-01  y = SDR + 21.23
5           2.081e+00  3.527e-02  y = (GDP * SDR) + 20.934
7           2.040e+00  9.940e-03  y = (GDP * (GDP + SDR)) + 19.778
9           2.020e+00  5.032e-03  y = (GDP * 8.0332) + (12.024 - (SDR * -1.1455))
11          2.020e+00  2.980e-07  y = 12.015 - ((GDP * -9.1902) - ((SDR - GDP) * 1.1446))
15          2.017e+00  3.282e-04  y = (12.839 - ((GDP * -5.6884) * GDP)) - ((GDP + SDR) / (G...
                                      DP / -1.2289))
───────────────────────────────────────────────────────────────────────────────────────────────────
[2026-03-04 20:36:29]
 DEBUG AFTER PREDICT - PySR_niter20:
  type(y_pred_train): <class 'numpy.ndarray'>

[2026-03-04 20:36:29]
 MODEL QUALITY METRICS: PySR_niter20
Training Set Performance:
  MSE:  2.233309
  RMSE:

...

  - outputs\20260304_204148_gEmKdM\hall_of_fame.csv

Expressions evaluated per second: 9.680e+04
Progress: 537 / 2480 total iterations (21.653%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.791e+00  0.000e+00  y = 25.194
3           1.057e+00  4.857e-01  y = Elast_ESD_Driver * 25.184
5           3.932e-01  4.942e-01  y = SDR + (Elast_ESD_Driver * 21.236)
7           3.403e-01  7.231e-02  y = ((Elast_ESD_Driver / 0.049588) + GDP) + SDR
9           2.823e-01  9.339e-02  y = (GDP / 0.24335) + ((Elast_ESD_Driver / 0.059408) + SDR...
                                      )
11          1.846e-01  2.124e-01  y = ((Elast_ESD_Driver / 0.05227) + ((SDR * GDP) - 0.20644...
                                      )) * 1.0868
13          1.290e-01  1.793e-01  y = (((SDR + Elast_ESD_Driver) +

[ Info: Started!



Expressions evaluated per second: 9.660e+04
Progress: 1088 / 3100 total iterations (35.097%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.791e+00  0.000e+00  y = 25.194
3           1.057e+00  4.857e-01  y = Elast_ESD_Driver * 25.184
5           3.932e-01  4.942e-01  y = SDR - (Elast_ESD_Driver * -21.236)
7           2.836e-01  1.634e-01  y = (SDR * GDP) - (-21.23 * Elast_ESD_Driver)
9           1.966e-01  1.832e-01  y = ((GDP + 0.092211) * SDR) - (Elast_ESD_Driver * -20.578...
                                      )
11          1.314e-01  2.015e-01  y = ((Other_ESD_Drivers * (SDR * GDP)) - (-21.23 * Elast_E...
                                      SD_Driver)) - 0.26028
13          1.284e-01  1.158e-02  y = (((Other_ESD_Drivers * (SDR * GDP)) - (-21.23 * Elast_...
            

[ Info: Final population:
[ Info: Results saved to:


[2026-03-04 20:42:55]
 DEBUG AFTER PREDICT - PySR_niter100:
  type(y_pred_train): <class 'numpy.ndarray'>

[2026-03-04 20:42:55]
 MODEL QUALITY METRICS: PySR_niter100
Training Set Performance:
  MSE:  0.072556
  RMSE: 0.269363
  MAE:  0.208445
  R²:   0.974005

Test Set Performance:
  MSE:  0.084609
  RMSE: 0.290876
  MAE:  0.226436
  R²:   0.972713

Overfitting Check:
  Train-Test MSE Ratio: 0.8575
  Train-Test R² Diff:   0.0013

TOP 5 EQUATIONS (PySR):
Could not retrieve equations: "['size'] not in index"

[2026-03-04 20:42:55]
 GENERATING PARAMETER SWEEP VISUALIZATIONS

Saved heatmap for Magpie
Saved heatmap for PySR
Saved line plots
Saved top parameters bar chart
Saved results CSV to parameter_sweep_analysis/parameter_sweep_results.csv

BEST PARAMETERS BY MODEL TYPE

Magpie:
  Best maxevals: 10000
  Best initevals: 4000
  Test MSE: 0.028191
  Test R²: 0.990908
  Scenario: BASE_SSP2, Target: GCost, Subset: all

PySR:
  Best max_evals: 10000
  Best niterations: 40
  Test MSE: 0.07531

# Bar Plots
The bar plot shows which features are most important to the model. It plots the absolute mean shap plot. each bar gives the absolute mean shap value of each feature. There is a shap value for each instance of each feature. The absolute value is taken and the mean is calculated. We take the absolute value as we do not want positive and negative values to offset eachother.
Values that have made large positive or negative contributions will have a large shap value. These are the features that have made a significant contribution to the models predictions. This is a useful measure of feature importance. 


# Beeswarm plot
This helps to visualise all shap values. On the Y-axis the values are grouped by all the features.The colour of the points is determined by the feature value. Higher values are red. On the X axis is the shap values. This can help to highlight important relationships. It shows which values have large positive or large negative values. The beeswarm plot can help understand the nature of relationships between feature variables - for example, in our beeswarm plot, for almost all variables, as the feature value increases, so does the shap values (except for wind_inv_cost)

# Dependence plots 
Not completed yet

# Violin Plot
Gives the same insights as the beeswarm plot

# Heatmap
X-axis shows a tick for each instance of data
Y-axis gives the model feature
The line above each instance is coloured by the shap value for that feature. 
The f(x) line (the line at the top of the graph) gives the predicted y value for that instance. 
The faint grey line behind the f(x) plot is the baseline or expected value of the models predictions
The bar on the right gives the mean shap values (these are the same as what we saw in the bar plot). 
The heatmap is a plot of every shap value, except we focus on shap values and groups of instances. 
By default the instances are ordered using a hierarchical clustering algorithm. optionally: We can change it to be ordered by the values of one of the features

## Interpreting plots
Scenario: BASE_SSP2
    - This scenario means that there is no upper limit on temperature increase . There is also no delay as there is no action
Target: GSupply
Subset: All
# Heatmap
This plot only shows 161 instances
'Elast_ESD_Driver' is given as the most influential feature in predicting GSupply. It seems to have both a strong positive and negative influence on the target variable

# Violin plot
Again, 'Elast_ESD_Driver' is the most influential in predicting 'GSupply' in both a positive and negative direction. Lower feature values correlate with lower 'Elast_ESD_Driver' values. The same can be said for 'GDP' and 'Other_ESD_Drivers'. 

# Scatter plot
This is a dependence scatter plot. it shows the effect that a single feature has on the prediction. Each dot is a single prediction of GSupply. The X-axis is the value of the feature. 
The Y-axis is the shap value of the feature. This represents how much knowing that features value changes the output of the model for that samples prediction
The light grey area in the background is a histogram showing the distribution of the data values. 
The histogram plots how many times a certain value appears for a feature in the dataset

We can see that with an increase in SHAP values for GDP, Other_ESD_Drivers and Elast_ESD_Driver, there is an increase in GSupply. There is also a slightly negative trend in shap values for Oil_Gas_Poten and Wind_Inv_Cost as GSupply increases

# Bar Plot
The bar plot shows a similar story



# Interpreting plots:
Scenario: 2C_SSP2_DA30
    This scenario means that there is a 2 degree celcius limit on temperature increase, with a 25 year delay action
Target: GSupply
Subset: All

# Heatmap
Clim_Sens is the most impactful feature on the model. Increase in Clim_Sens is associated with increase in GSupply.
# Violin plot and beeswarm
The more negative the shap value for Clim_sens leads to a more negative feature value.
There is a larger number of Shap values in the negative range for Clim_Sens than in the positive range. 
The more positive the Shap value for SDR, the more negative the feature value
# Bar Plot
Shows that Clim_Sens has the greatest mean absolute shap value, followed by Elast_ESD_Driver, SRD and GDP
# Scatter plot
This shows that as the value of GDP, Elast_ESD_Driver and Clim_Sens increases, so does the SHAP value. And as SDR increases, the shap values decrease




# Interpreting plots:
Scenario: 2C_SSP2_DA30
    This scenario means that there is a 2 degree celcius limit on temperature increase, with a 25 year delay action
Target: GConsumption
Subset: All

# Heatmap
Both Clim_Sens and Elast_ESD_Driver are closely tied as being most impactful to the prediction of GConsumption for the 2C_SSP2_DA30 scenario. There is no general pattern for 
# Violin plot and beeswarm
# Bar Plot
# Scatter plot

# The Background data:
The key behind shapley values is that the features act as a "team", and the prediction is fairly distributed among them. The algorithm computes what predictions different coalitions (subsets of features) would receive. Since most ML models require all data to be filled in, missing values are sampled from the background data. Different background datasets can make different shap interpretations. What ever we choose as the background dataset influences for which data points we get predictions, and therefore what the interpretation is. for example, if we choose the background data based on a certain feature, this feature will no longer explain variation. Analogy: Height is an influential factor on performance in playing basketball. But within the NBA, where most players are tall, height no longer explains differences in performance. This is similar to collider bias or selection bias. Whether it is a problem or not depends on the purpose of the analysis - the example given for this is when comparing a wine against only high alcohol wines.

# shap.Explainer:
- Uses Shapley values to explain any ML model or python function.
- It is the primary explainer interface for the SHAP library. It takes any combo of model and masker and returns a callable subclass object that implements the particular estimation algorithm that was chosen.
    - Masker: explainer takes 'masker' as a parameter. The function is used to 'mask' out hidden features of the form masked_args = masker(*model_args, mask=mask). Background data matrices can be passed instead of a function, and the matrix will be used for masking. This is a shortcut for standard masking.

# Kernel Explainer:
Kernel SHAP is a model agnostic explainer. It is a method that uses special weighted linear regression to compute the importance of each feature. The computed importance values are Shapley values adn also coefficients from a local linear regression.



In [32]:
cv = 5
#results = run_everything(Xy, cv)
results_cols = ['Target', 'Scenario', 'xvar_subset_code', 'Regression Model', 'Fold', 'Train Score', 'Test Score']
results_df = pd.DataFrame(results, columns=results_cols)
results_df['Scenario / Variables'] = results_df['Scenario'] + ' / ' + results_df['xvar_subset_code']
#results_df.to_csv(f'../outputs/results_cv5_EN_RF_2xGP_clim_sens_filter_sqrt.csv')

NameError: name 'results' is not defined

Check that the results are the shape we expect:

In [ ]:
results_df.head()

In [ ]:
results_df.shape

In [ ]:
5 * 4 * 2 * 4 * 5 # 5 scenarios (incl ALL), 4 targets, 2 xvar subsets (variant in the case of ALL), 4 regression methods, 5 cv folds

# Statistics

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
models = ['ElasticNet', 'RandomForestRegressor', 'PySRRegressor']
for model in models:
    sub = results_df[(results_df['Regression Model'] == model) & (results_df['Scenario'] != 'ALL')]['Test Score']
    print(f'{model} mean {sub.mean():.2f} std {sub.std():.1f}')
    for model2 in models:
        if model2 != model:
            sub2 = results_df[(results_df['Regression Model'] == model2) & (results_df['Scenario'] != 'ALL')]['Test Score']
            print(f'{model} {model2} {ttest_ind(sub, sub2)}')

In [ ]:
for model in models:
    sub1 = results_df[(results_df['Regression Model'] == model) & (results_df['xvar_subset_code'] == 'small')]['Test Score']
    sub2 = results_df[(results_df['Regression Model'] == model) & (results_df['xvar_subset_code'] == 'all')]['Test Score']
    print(f'{model} small {sub1.mean():.2f} v all {sub2.mean():.2f}: {ttest_ind(sub1, sub2)}')

In [ ]:
# plot the individual scenario regression results
for target in targets:

    # we exclude GPAFSRegressor for now
    tmp = results_df[(results_df['Target'] == target) & (results_df['Regression Model'] != 'GPAFSRegressor') 
                     & (results_df['Scenario'] != 'ALL')]

    sns.stripplot(tmp, x='Scenario / Variables', y='Train Score', hue='Regression Model');
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Train $R^2$ on {target}')
    plt.tight_layout()
    plt.savefig(f'../outputs/results_boxplot_cv5_{target}_train_clim_sens_filter_sqrt.pdf')
    plt.close()

    sns.stripplot(tmp, x='Scenario / Variables', y='Test Score', hue='Regression Model');
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Test $R^2$ on {target}')
    plt.tight_layout()
    plt.savefig(f'../outputs/results_boxplot_cv5_{target}_test_clim_sens_filter_sqrt.pdf')
    plt.close()


In [ ]:
# plot the *unified* scenario regression results
for target in targets:

    # we exclude GPAFSRegressor for now
    tmp = results_df[(results_df['Target'] == target) & (results_df['Regression Model'] != 'GPAFSRegressor')
                     & (results_df['Scenario'] == 'ALL')]

    sns.stripplot(tmp, x='Scenario / Variables', y='Train Score', hue='Regression Model');
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Train $R^2$ on {target}')
    plt.tight_layout()
    plt.savefig(f'../outputs/results_boxplot_cv5_{target}_unified_scenarios_train_clim_sens_filter_sqrt.pdf')
    plt.close()

    sns.stripplot(tmp, x='Scenario / Variables', y='Test Score', hue='Regression Model');
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Test $R^2$ on {target}')
    plt.tight_layout()
    plt.savefig(f'../outputs/results_boxplot_cv5_{target}_unified_scenarios_test_clim_sens_filter_sqrt.pdf')
    plt.close()


# Interpretation of models

## Variable importance and equation interpretation

In this section, we'll use a single train-test split and do a single run (per scenerio) for each of EN, RF, and SR
as our goal will not be a comparison of performance but an investigation of variable importance.

We include each scenario and all variables, not including the scenario-encoding variables.

RF gives variable importance directly.

EN does not, but we use abs(coefficient_i) * std(x_i)

SR we'll just store the equations together with their loss and complexity values.

In [ ]:
scenarios = ['BASE_SSP2', '1p5c_OS_SSP2', '2C_SSP2', '2C_SSP2_DA30']

import sympy_latex

for scenario in scenarios:
    Xy_tmp = Xy[Xy['Scenario'] == scenario]
    X = Xy_tmp[xvar_subsets['all']]
    for target in targets:
        y = Xy_tmp[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

        for m in [#RandomForestRegressor(), 
                  #ElasticNet(), 
                  PySRRegressor(unary_operators=["sqrt"]), 
                  #GPAFSRegressor(20000, 4000, 0.5, 30, 5, "symbolic_regression.bnf", 0.3)]:
        ]:

            #m.fit(X_train, y_train)
            #m.score(X_test, y_test)

            if m.__class__.__name__ == 'RandomForestRegressor':
                # importance is given directly by the method
                imps = list(zip(m.feature_names_in_, m.feature_importances_))
                imps.sort(key=lambda x: x[1], reverse=True)
                imps = pd.DataFrame(imps, columns=['Feature', 'Importance'])
                # print, write importances to csv and to tex
                print(imps)
                imps.to_csv(f'../outputs/importances_rf_{scenario}_{target}_clim_sens_filter_sqrt.csv')
                imps['Feature'] = imps['Feature'].str.replace('_', ' ')
                imps.to_latex(f'../outputs/importances_rf_{scenario}_{target}_clim_sens_filter_sqrt.tex', index=False)     
            elif m.__class__.__name__ == 'ElasticNet':
                # importance is abs(coef) * std deviation of the var
                X_std = X_train.std(axis=0)
                imps = np.abs(m.coef_) * X_std
                imps = list(zip(m.feature_names_in_, imps))
                imps.sort(key=lambda x: x[1], reverse=True)
                imps = pd.DataFrame(imps, columns=['Feature', 'Importance'])
                # print, write importances to csv and to tex
                print(imps)
                imps.to_csv(f'../outputs/importances_en_{scenario}_{target}_clim_sens_filter_sqrt.csv')
                imps['Feature'] = imps['Feature'].str.replace('_', ' ')
                imps.to_latex(f'../outputs/importances_en_{scenario}_{target}_clim_sens_filter_sqrt.tex', index=False)
            elif m.__class__.__name__ == 'PySRRegressor' or m.__class__.__name__ == 'GPAFSRegressor':

                if m.__class__.__name__ == 'PySRRegressor': method = 'sr'
                if m.__class__.__name__ == 'GPAFSRegressor': method = 'magpie'

                # print, write original equations to csv
                #print(m.equations_)
                #m.equations_.to_csv(f'../outputs/equations_{method}_{scenario}_{target}_clim_sens_filter_sqrt.csv')
                # eqns = m.equations_

                eqns = pd.read_csv(f'../outputs/equations_{method}_{scenario}_{target}_clim_sens_filter_sqrt.csv')

                # write to latex table, but process first, and only useful columns
                eqns = sympy_latex.process_equations_latex(eqns)
                eqns = eqns[['vars', 'consts', 'complexity', 'loss', 'equation']]
                #eqns['equation'] = eqns['equation'].str.replace('_', ' ')
                eqns.to_latex(f'../outputs/equations_{method}_{scenario}_{target}_3cols_clim_sens_filter_sqrt.tex', index=False)
